In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

ollama_api_key = os.getenv("OLLAMA_API_KEY")

print("OLLAMA API KEY FOUND:", bool(ollama_api_key))

OLLAMA API KEY FOUND: True


In [3]:
# !pip install -q langchain-ollama langchain-core requests python-dotenv

In [4]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests


**Creating Tool**

In [5]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [6]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [7]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Given 2 numbers a and b this tool returns their product
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


**tool binding**

In [9]:
import os
from dotenv import load_dotenv
from langchain_ollama import ChatOllama

load_dotenv()

ollama_api_key = os.getenv("OLLAMA_API_KEY")

llm = ChatOllama(
    model="gpt-oss:120b",
    base_url="https://ollama.com",
    temperature=0.2,
    client_kwargs={
        "headers": {
            "Authorization": f"Bearer {ollama_api_key}"
        }
    }
)

In [10]:
# Send a simple message to the Ollama Cloud LLM
response = llm.invoke("hi")

# Print only the text generated by the LLM
print(response.content)

Hello! 👋 How can I assist you today?


In [11]:
# Bind the multiply tool to the LLM so it can call the tool when required
llm_with_tools = llm.bind_tools([multiply])

**Tool Calling**

In [43]:
# Send a normal message to the LLM with the multiply tool available
response = llm_with_tools.invoke("Hi, How are you")

print(response.content)

Hello! I'm doing great, thank you for asking. How can I assist you today?


In [15]:
# Create a human message containing the user's multiplication request
query = HumanMessage(content="Can you multiply 4 with 1000?")

In [16]:
# Create a list containing the user's HumanMessage query
messages = [query]

In [17]:
messages

[HumanMessage(content='Can you multiply 4 with 1000?', additional_kwargs={}, response_metadata={})]

In [18]:
# Send the messages to the LLM with the multiply tool available
result = llm_with_tools.invoke(messages)

In [19]:
# Add the LLM's response (including any tool call) to the messages list
messages.append(result)

In [20]:
messages

[HumanMessage(content='Can you multiply 4 with 1000?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-09-21T10:02:21.485486085Z', 'done': True, 'done_reason': 'stop', 'total_duration': 318419508, 'load_duration': None, 'prompt_eval_count': 141, 'prompt_eval_duration': None, 'eval_count': 50, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'}, id='lc_run--01a0c36a-a65b-70e1-9556-fe331c005f6a-0', tool_calls=[{'name': 'multiply', 'args': {'a': 4, 'b': 1000}, 'id': 'b8393a0f-73de-457e-b031-e2b49890ce97', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 141, 'output_tokens': 50, 'total_tokens': 191})]

In [21]:
# Execute the multiply tool using the tool call generated by the LLM
tool_result = multiply.invoke(result.tool_calls[0])

In [22]:
# Print the result returned by the multiply tool
print(tool_result)

content='4000' name='multiply' tool_call_id='b8393a0f-73de-457e-b031-e2b49890ce97'


In [23]:
# Add the tool's result to the messages list
messages.append(tool_result)

In [24]:
messages

[HumanMessage(content='Can you multiply 4 with 1000?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-09-21T10:02:21.485486085Z', 'done': True, 'done_reason': 'stop', 'total_duration': 318419508, 'load_duration': None, 'prompt_eval_count': 141, 'prompt_eval_duration': None, 'eval_count': 50, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'}, id='lc_run--01a0c36a-a65b-70e1-9556-fe331c005f6a-0', tool_calls=[{'name': 'multiply', 'args': {'a': 4, 'b': 1000}, 'id': 'b8393a0f-73de-457e-b031-e2b49890ce97', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 141, 'output_tokens': 50, 'total_tokens': 191}),
 ToolMessage(content='4000', name='multiply', tool_call_id='b8393a0f-73de-457e-b031-e2b49890ce97')]

In [32]:
# Send the complete conversation back to the LLM and get the final text response
llm_with_tools.invoke(messages).content

'Sure! \\(4 \\times 1000 = 4000\\).'